# CropSense Module 2 — Random Forest improvement with field-separated evaluation

**Required architecture:** Random Forest. **No new metrics are claimed before execution.**

The archived notebook's final run reports 267/400 correct (66.75%), macro F1 approximately 0.537,
healthy recall 7/67, and nine severe→healthy mistakes. Its 2,000-image sample depended on unsorted filesystem order.
This run uses all available pairs in sorted order and separates field identifiers across train/validation/test.
It therefore establishes a **new protocol baseline**; a score change from the historical random-tile split is not
by itself an accuracy improvement. Fields are inferred from the filename prefix before the first underscore;
this convention is recorded and still needs dataset-level provenance confirmation.

The six raw NDVI statistics and heuristic stress-label rules are preserved. No label masks become model inputs.
Labels are not independent agronomist diagnoses; water exclusion and anomaly thresholds remain limitations.


In [ ]:
PROJECT_MODULE='cropsense-module-2'
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn==1.6.1','boto3'],check=True)


In [ ]:
import os, sys, json, hashlib, platform, subprocess, time
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, log_loss)

SEED = 42
np.random.seed(SEED)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = Path('/kaggle/working') / PROJECT_MODULE / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

def write_json(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

write_json('environment.json', {'run_id':RUN_ID, 'python':sys.version, 'platform':platform.platform(),
    'packages':{d.metadata['Name']:d.version for d in metadata.distributions()},
    'seed':SEED, 'notebook_status':'executed; results below are generated, not pre-filled'})
results = []

def evaluate(name, split, y_true, probabilities, classes, sample_ids):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    assert probabilities.shape == (len(y_true), len(classes))
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-4)
    pred = probabilities.argmax(axis=1)
    labels = np.arange(len(classes))
    report = classification_report(y_true, pred, labels=labels, target_names=classes,
                                   output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, pred, labels=labels)
    present = np.unique(y_true)
    row = {'model':name, 'split':split, 'n':len(y_true),
           'accuracy':float(accuracy_score(y_true,pred)),
           'balanced_accuracy':float(balanced_accuracy_score(y_true,pred)),
           'macro_f1_all_classes':float(f1_score(y_true,pred,labels=labels,average='macro',zero_division=0)),
           'macro_f1_present_classes':float(f1_score(y_true,pred,labels=present,average='macro',zero_division=0)),
           'weighted_f1':float(f1_score(y_true,pred,average='weighted',zero_division=0)),
           'log_loss':float(log_loss(y_true, probabilities, labels=labels))}
    # Wilson interval is descriptive; correlated field tiles/weeks violate independent-sample assumptions.
    n=len(y_true); p=row['accuracy']; z=1.96
    center=(p+z*z/(2*n))/(1+z*z/n)
    radius=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
    row['accuracy_wilson95_descriptive']=[float(center-radius),float(center+radius)]
    if classes == ['healthy','moderate_stress','severe_stress']:
        row['healthy_recall']=float(report['healthy']['recall'])
        row['severe_to_healthy_count']=int(cm[2,0])
        row['severe_to_healthy_rate']=float(cm[2,0]/max(cm[2].sum(),1))
    prefix=f'{name}__{split}'
    write_json(prefix+'_metrics.json', {'summary':row,'per_class':report,'class_order':classes,
                                      'confusion_matrix':cm.tolist()})
    pd.DataFrame(report).T.to_csv(OUT/(prefix+'_classification_report.csv'))
    pd.DataFrame(cm,index=classes,columns=classes).to_csv(OUT/(prefix+'_confusion_counts.csv'))
    frame=pd.DataFrame({'sample_id':sample_ids,'true_index':y_true,'pred_index':pred})
    for i,c in enumerate(classes): frame['prob_'+c]=probabilities[:,i]
    frame.to_csv(OUT/(prefix+'_predictions.csv'),index=False)
    fig,axes=plt.subplots(1,2,figsize=(max(12,len(classes)*.65),max(5,len(classes)*.35)))
    norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
    for ax,data,title in zip(axes,[cm,norm],['Counts','Recall by true class']):
        im=ax.imshow(data,cmap='Blues');fig.colorbar(im,ax=ax,fraction=.04)
        ax.set(xticks=labels,yticks=labels,xticklabels=classes,yticklabels=classes,
               xlabel='Predicted',ylabel='True',title=title)
        ax.tick_params(axis='x',rotation=90)
        if len(classes)<=6:
            for r in labels:
                for c in labels: ax.text(c,r,f'{data[r,c]:.2f}' if data is norm else str(data[r,c]),ha='center',va='center')
    fig.suptitle(prefix);fig.tight_layout();fig.savefig(OUT/(prefix+'_confusion.png'),dpi=150);plt.show();plt.close(fig)
    print(pd.DataFrame(report).T.to_string());print(row)
    results.append(row)
    pd.DataFrame(results).to_csv(OUT/'results_summary.csv',index=False)
    return row

def save_history(name, history):
    pd.DataFrame(history.history).to_csv(OUT/(name+'_history.csv'),index_label='epoch')
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for ax,metric in zip(axes,['loss','accuracy']):
        for key in [metric,'val_'+metric]:
            if key in history.history: ax.plot(np.arange(1,len(history.history[key])+1),history.history[key],label=key)
        ax.set(xlabel='Epoch',title=metric);ax.legend()
    fig.tight_layout();fig.savefig(OUT/(name+'_learning_curves.png'),dpi=150);plt.show();plt.close(fig)

def finish(model_file, contract, decision):
    write_json('model_contract.json', dict(contract, artifact=Path(model_file).name,
                                         sha256=sha256(model_file),run_id=RUN_ID))
    write_json('decision.json',decision)
    pd.DataFrame(results).to_csv(OUT/'results_summary.csv',index=False)
    lines=['# '+PROJECT_MODULE+' — executed experiment report', '', 'Run: '+RUN_ID,
           '', '## Results', '', pd.DataFrame(results).to_csv(index=False),
           '## Decision', '', json.dumps(decision,indent=2),
           '', '## Reproduction', 'See environment.json, config.json, split_manifest.csv, model_contract.json, and this executed notebook.',
           '', '## Limitations', contract.get('limitations','')]
    (OUT/'REPORT.md').write_text('\n'.join(lines),encoding='utf-8')
    import shutil
    archive=shutil.make_archive(str(OUT.parent/(RUN_ID+'_evidence')),'zip',OUT)
    print('Download evidence:',archive)
    print('Candidate export:',model_file,'SHA256:',sha256(model_file))


## 1. Data acquisition and deterministic feature extraction
Download the public Agriculture-Vision mini-scale archive directly inside Kaggle. Save its hash,
every accepted sample's features and label, and every exclusion or decoding error.
Training and dashboard use raw `(NIR − red)/(NIR + red + 1e-8)`, not rescaled NDVI.


In [ ]:
import boto3,tarfile
from botocore import UNSIGNED
from botocore.client import Config
from PIL import Image
bucket='intelinair-data-releases';key='agriculture-vision/cvpr_paper_2020/Dataset/data2017_miniscale.tar.gz'
archive=Path('/kaggle/working/data2017_miniscale.tar.gz')
base=Path('/kaggle/working/data2017_miniscale')
if not (base/'field_images/rgb').exists():
    if not archive.exists():
        boto3.client('s3',config=Config(signature_version=UNSIGNED)).download_file(bucket,key,str(archive))
    with tarfile.open(archive) as tar: tar.extractall('/kaggle/working',filter='data')
write_json('data_provenance.json',{'bucket':bucket,'key':key,'archive_sha256':sha256(archive),
    'retrieved_utc':datetime.now(timezone.utc).isoformat(),'source':'Agriculture-Vision 2017 miniscale'})
FEATURES=['ndvi_mean','ndvi_std','ndvi_min','ndvi_max','ndvi_p25','ndvi_p75']
CLASSES=['healthy','moderate_stress','severe_stress']
severe=['drydown','nutrient_deficiency','storm_damage']
moderate=['double_plant','planter_skip','endrow','weed_cluster']
exclude=['water','waterway'];all_labels=severe+moderate+exclude
assert all((base/'field_labels'/x).is_dir() for x in all_labels)
records=[];rejected=[]
paths=sorted((base/'field_images/rgb').glob('*.jpg'))
assert len(paths)>1000
for i,rgb_path in enumerate(paths):
    nir_path=base/'field_images/nir'/rgb_path.name
    try:
        with Image.open(rgb_path) as image: rgb=np.asarray(image.convert('RGB'),dtype=float)
        with Image.open(nir_path) as image:
            assert image.mode=='L',f'Unexpected NIR encoding {image.mode}'
            nir=np.asarray(image,dtype=float)
        assert nir.shape==rgb.shape[:2], 'Unaligned dimensions'
        red=rgb[:,:,0];ndvi=(nir-red)/(nir+red+1e-8)
        coverage={}
        for label in all_labels:
            mask_path=base/'field_labels'/label/(rgb_path.stem+'.png')
            if mask_path.exists():
                with Image.open(mask_path) as im: mask=np.asarray(im)
                assert mask.shape==ndvi.shape, 'Unexpected mask dimensions'
                coverage[label]=float(np.mean(mask>0)*100)
            else: coverage[label]=0.0
        if any(coverage[x]>15 for x in exclude):
            rejected.append({'filename':rgb_path.name,'reason':'water exclusion'});continue
        label='severe_stress' if any(coverage[x]>5 for x in severe) else (
              'moderate_stress' if any(coverage[x]>2 for x in moderate) else 'healthy')
        values=[ndvi.mean(),ndvi.std(),ndvi.min(),ndvi.max(),*np.percentile(ndvi,[25,75])]
        records.append(dict(filename=rgb_path.name,field_id=rgb_path.stem.split('_')[0],label=label,
                            **dict(zip(FEATURES,map(float,values))),
                            **{'coverage_'+k:v for k,v in coverage.items()}))
    except Exception as exc:
        rejected.append({'filename':rgb_path.name,'reason':f'{type(exc).__name__}: {exc}'})
    if i%500==0: print(i,'/',len(paths),'accepted',len(records),flush=True)
df=pd.DataFrame(records)
df.to_csv(OUT/'features_and_labels.csv',index=False)
pd.DataFrame(rejected,columns=['filename','reason']).to_csv(OUT/'excluded_samples.csv',index=False)
decode_errors=[x for x in rejected if x['reason']!='water exclusion']
assert not decode_errors, f'{len(decode_errors)} data errors: inspect excluded_samples.csv before training.'
print(df['label'].value_counts());print('Fields:',df.field_id.nunique())


## 2. Freeze disjoint field splits before training
60% / 20% / 20% approximate field split. Save sample IDs and field IDs. Every split must contain all classes.
No test data influences candidate choice. The dataset is larger than the historical 2,000-image subset.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
trainval_idx,test_idx=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(df,groups=df.field_id))
tv=df.iloc[trainval_idx]
train_local,val_local=next(GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED).split(tv,groups=tv.field_id))
parts={'train':tv.iloc[train_local].copy(),'validation':tv.iloc[val_local].copy(),'test':df.iloc[test_idx].copy()}
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    assert not (set(parts[a].field_id)&set(parts[b].field_id))
for split,part in parts.items():
    assert set(part.label)==set(CLASSES),f'{split} lacks a class; stop and revise protocol explicitly.'
    print(split,len(part),'fields',part.field_id.nunique(),part.label.value_counts().to_dict())
pd.concat([p[['filename','field_id','label']].assign(split=s) for s,p in parts.items()]).to_csv(OUT/'split_manifest.csv',index=False)
config={'seed':SEED,'feature_order':FEATURES,'class_order':CLASSES,'group':'filename prefix before first underscore',
        'train_fraction':.6,'validation_fraction':.2,'test_fraction':.2,
        'label_thresholds_percent':{'severe':5,'moderate':2,'water_exclusion':15},
        'selection':'maximize validation macro F1 while healthy recall does not fall and severe-to-healthy rate does not rise',
        'candidate_configs':[{'n_estimators':200,'class_weight':'balanced'},
             {'n_estimators':400,'class_weight':'balanced','min_samples_leaf':2,'max_features':1.0},
             {'n_estimators':400,'class_weight':'balanced_subsample','min_samples_leaf':4,'max_depth':20},
             {'n_estimators':400,'class_weight':{'healthy':1.8,'moderate_stress':1.0,'severe_stress':2.5},'min_samples_leaf':2},
             {'n_estimators':400,'class_weight':{'healthy':2.5,'moderate_stress':1.0,'severe_stress':3.0},'min_samples_leaf':2},
             {'n_estimators':400,'class_weight':{'healthy':1.5,'moderate_stress':1.0,'severe_stress':3.0},'min_samples_leaf':4},
             {'n_estimators':400,'class_weight':'balanced','min_samples_leaf':2,'max_depth':14}]}
write_json('config.json',config)


## 3. Fit the required RF architecture and select on validation only
The first candidate is the original 200-tree configuration retrained on the new training split.
Healthy recall and severe→healthy errors are explicit checks, so aggregate accuracy cannot hide the weakest class.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib
tr=parts['train'];val=parts['validation'];candidates={};validation=[]
for i,spec in enumerate(config['candidate_configs']):
    name='rf_baseline' if i==0 else f'rf_candidate_{i}'
    model=RandomForestClassifier(**spec,random_state=SEED,n_jobs=-1)
    start=time.perf_counter();model.fit(tr[FEATURES],tr.label)
    assert list(model.classes_)==CLASSES
    row=evaluate(name,'validation',val.label.map(CLASSES.index).to_numpy(),model.predict_proba(val[FEATURES]),CLASSES,val.filename.tolist())
    row['training_seconds']=time.perf_counter()-start
    candidates[name]=model;validation.append(row)
baseline=validation[0]
eligible=[r for r in validation if r['healthy_recall']>=baseline['healthy_recall'] and
          r['severe_to_healthy_rate']<=baseline['severe_to_healthy_rate']]
chosen=max(eligible,key=lambda r:(r['macro_f1_all_classes'],r['accuracy']))['model']
write_json('selection_before_test.json',{'selected':chosen,'validation':validation,'eligible':[r['model'] for r in eligible]})
print('Selected:',chosen)


## 4. Final held-out comparison and export
Evaluate only the baseline and the preselected RF on held-out fields. Store per-image predictions,
per-class metrics, both confusion matrices, feature importances, training time and a verified artifact.
No deployment replacement happens inside this notebook.


In [ ]:
test=parts['test'];yt=test.label.map(CLASSES.index).to_numpy();test_rows={}
for name in dict.fromkeys(['rf_baseline',chosen]):
    test_rows[name]=evaluate(name,'test',yt,candidates[name].predict_proba(test[FEATURES]),CLASSES,test.filename.tolist())
model=candidates[chosen]
pd.Series(model.feature_importances_,index=FEATURES,name='importance').sort_values(ascending=False).to_csv(OUT/'feature_importances.csv')
export=OUT/'module2_rf_candidate.pkl';joblib.dump(model,export)
reloaded=joblib.load(export)
np.testing.assert_allclose(model.predict_proba(test[FEATURES]),reloaded.predict_proba(test[FEATURES]),atol=1e-12,rtol=1e-12)
selected=test_rows[chosen];base_result=test_rows['rf_baseline']
finish(export,{'feature_order':FEATURES,'class_order':CLASSES,'preprocessing':'raw float64 NDVI; six unscaled statistics',
    'scikit_learn_version':metadata.version('scikit-learn'),
    'limitations':'Heuristic anomaly-mask labels; inferred field identifiers; no agronomist ground truth; new split is not comparable to historical random-tile accuracy.'},
    {'selected':chosen,'reload_predictions_match':True,
     'test_macro_f1_improved':selected['macro_f1_all_classes']>base_result['macro_f1_all_classes'],
     'test_healthy_recall_not_worse':selected['healthy_recall']>=base_result['healthy_recall'],
     'test_severe_to_healthy_rate_not_worse':selected['severe_to_healthy_rate']<=base_result['severe_to_healthy_rate'],
     'deployment':'candidate only; review held-out metrics and integration checks before promotion'})
